# Smart Waste Segregation using Computer Vision
### Industrial Training Project - B.Tech Computer Engineering

This notebook documents the training, evaluation, and inference workflow for the waste segregation system.

**Target Classes (5):**
- `0: plastic`
- `1: paper`
- `2: metal`
- `3: glass`
- `4: organic`

In [1]:
# 1. Import necessary libraries
import os
import cv2
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("Libraries imported successfully.")

## 2. Inspect Dataset Configuration (`data.yaml`)

In [2]:
yaml_path = "../data.yaml"
if os.path.exists(yaml_path):
    with open(yaml_path, "r") as f:
        cfg = yaml.safe_load(f)
    print("Classes count:", cfg.get('nc'))
    print("Class labels:", cfg.get('names'))
else:
    print("data.yaml not found at", yaml_path)

## 3. Load YOLOv8 Model
We use `yolov8n.pt` (nano architecture) for efficient training and inference.

In [3]:
# Load base pretrained model
model = YOLO("yolov8n.pt")
print("Model loaded:", model)

## 4. Train the Model
Fine-tune the YOLOv8n model on the custom waste dataset for 50 epochs.

In [4]:
# Train model (data.yaml specifies images/train, images/val)
results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="waste_seg"
)

## 5. Model Validation
Evaluate model performance on the validation set using the best saved weights.

In [5]:
# Validate model
metrics = model.val()
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

## 6. Test Inference & Visualization

In [6]:
# Run test inference on sample image
test_img = "../assets/sample_images/sample_waste.png"
if os.path.exists(test_img):
    res = model(test_img)
    annotated = res[0].plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Smart Waste Detection Output")
    plt.show()
else:
    print("Test image not found at", test_img)